### ------ ENGLISH ------

### Project 1: Retail Sales Performance and Profitability Analysis

**Technologies used:** SQL, Python (Pandas), Power BI

### Notebook Objective

The starting dataset, Superstore Sales from Kaggle, is a flat CSV file.

The objective of this notebook is to clean, explore and transform the dataset into a simple star schema in order to simulate a more realistic business analytics environment.

The final model is composed of the following tables:

- **Dim_Customers**: customer master data and geographical information.
- **Dim_Products**: product catalog, categories and sub-categories.
- **Dim_Date**: calendar table with day, month, quarter and year details.
- **Fact_Sales**: fact table containing transactions, sales, discounts and profit values.

#### Notes

- Code lines that generate large table outputs are commented out after use in order to keep the notebook more readable.

- For a detailed summary of the process, please refert to 'data_cleaning_notes.md'

### ------ ITALIANO ------

### Progetto 1: Analisi delle Performance di Vendita e Redditività Retail
**Tecnologie utilizzate:** SQL (SQLite), Python (Pandas), Power BI

### Obiettivo di questo notebook:
Il dataframe di partenza (Superstore Sales da Kaggle) è un file CSV piatto. Per ottimizzare le performance e simulare un ambiente aziendale, i dati verranno normalizzati, esplorati e suddivisi in uno "Schema a Stella" composto dalle seguenti tabelle:

- **Dim_Customers**: Anagrafica clienti.
- **Dim_Products**: Catalogo prodotti e categorie.
- **DIM_Date**: Dettaglio di giorno / mese / quadrimestre / anno
- **Fact_Sales**: Tabella dei fatti contenente transazioni, vendite, sconti e profitti.


#### Note
- Le linee di codice che mandano esplosioni di tabelle a terminale, vengono inibite con # dopo l'utilizzo, in modo da rendere il programma più leggibile

- Per un sommario dettagliato del processo, fare riferimento al file 'data_cleaning_notes.md'

In [65]:
import os
import pandas as pd

# ------ funzione directory dinamica e lettura .csv ------ #

def open_csv (file, nome, *cartelle):
    csv_path = os.path.join("..", *cartelle, file)
    df = pd.read_csv(csv_path)
    df.attrs["name"] = nome
    return df


In [66]:
# ------ SEZIONE CHIAMATA FUNZIONI ------ #

df_superstore = open_csv("superstore.csv", "df_superstore", "data", "raw")
df_superstore.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  object 
 6   Customer Name   10194 non-null  object 
 7   Segment         10194 non-null  object 
 8   Country/Region  10194 non-null  object 
 9   City            10194 non-null  object 
 10  State/Province  10194 non-null  object 
 11  Postal Code     10194 non-null  object 
 12  Region          10194 non-null  object 
 13  Product ID      10194 non-null  object 
 14  Category        10194 non-null  object 
 15  Sub-Category    10194 non-null  object 
 16  Product Name    10194 non-null  object 
 17  Sales           10194 non-null 

La colonna Profit risulta str, forse a causa di uno o più typo.
Decido di convertire in numeri, andando poi a verificare il numero di righe rimanenti

In [67]:
# ------ CONVERSIONE COLONNA PROFIT IN NUMERICA ------ #
df_superstore["Profit"] = df_superstore["Profit"].str.strip().replace("", pd.NA)
df_superstore["Profit_numeric"] = pd.to_numeric(df_superstore["Profit"], errors="coerce")
print(f"Numero di righe perse dopo la conversione: {(df_superstore['Profit'].size) - (df_superstore['Profit_numeric'].count())}")


Numero di righe perse dopo la conversione: 1


Verifico l' esistenza di altri valori nulli

In [68]:
print(f"Valori mancanti totali in df_superstore: {df_superstore.isna().sum().sum()}")
print("\nValori mancanti per colonna df_superstore:")
#   print(df_superstore.isna().sum())

Valori mancanti totali in df_superstore: 1

Valori mancanti per colonna df_superstore:


Non essendoci altri valori nulli, ed essendo presente un solo valore non convertibile nella colonna "Profit" (meno dello 0,01% del dataframe), decido di rimuovere il record interessato. Per sicurezza opero su una copia del dataframe.

In [69]:
df_sup_normalized = df_superstore.copy()
df_sup_normalized = df_sup_normalized.drop(columns=["Profit"])
df_sup_normalized = df_sup_normalized.dropna()
df_sup_normalized = df_sup_normalized.rename(columns={"Profit_numeric": "Profit"})
#   df_sup_normalized.info()

Inserisco un ciclo di normalizzazione lower del testo per le colonne str

In [70]:
for colonna in df_sup_normalized.columns:
    if df_sup_normalized[colonna].dtype == "object":
        df_sup_normalized[colonna] = df_sup_normalized[colonna].str.lower()
    

Verifica di eventuali duplicati. Row ID dovrebbe identificare in modo univoco ogni record, mentre uno stesso Order ID potrebbe comparire su più righe nel caso di ordini contenenti prodotti differenti. Perciò verifico prima se esistono duplicati su Row ID per essere certo della bontà del dato. In caso negativo, escluderò la colonna dalla verifica e cercherò eventuali righe completamente duplicate.

In [71]:
duplicati_row_ID = df_sup_normalized.duplicated(subset=["Row ID"], keep=False).sum()
print(f"Righe duplicate colonna 'Row ID': {duplicati_row_ID}")

Righe duplicate colonna 'Row ID': 0


Row ID non contiene duplicati, lo escludo quindi dal controllo duplicati sul resto del dataframe

In [72]:
# creo lista di colonne da verificare
colonne_da_controllare = df_sup_normalized.columns.drop("Row ID")

# applico la lista come un subset
conteggio_duplicati = df_sup_normalized.duplicated(subset=colonne_da_controllare, keep=False).sum()
print(f"Numero di righe duplicate: {conteggio_duplicati}")

Numero di righe duplicate: 4


Il controllo dichiara 4 righe duplicate. Procedo a ispezionarle con una maschera.

In [73]:
maschera_duplicati = df_sup_normalized.duplicated(subset=colonne_da_controllare, keep=False)
df_sup_duplicati = df_sup_normalized[maschera_duplicati]
df_sup_duplicati.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
390,391,us-2023-150119,04/23/2023,04/27/2023,standard class,lb-16795,laurel beltran,home office,united states,columbus,...,43229,east,fur-ch-10002965,furniture,chairs,global leather highback executive chair with p...,281.372,2,0.3,-12.0588
391,392,us-2023-150119,04/23/2023,04/27/2023,standard class,lb-16795,laurel beltran,home office,united states,columbus,...,43229,east,fur-ch-10002965,furniture,chairs,global leather highback executive chair with p...,281.372,2,0.3,-12.0588
1698,1699,ca-2023-153623,11/24/2023,12/05/2023,standard class,jp-11135,james peterman,corporate,canada,st. john's,...,a0a,east,fur-fu-10002501,furniture,furnishings,nu-dell executive frame,99.120,8,0.0,35.4144
1699,1700,ca-2023-153623,11/24/2023,12/05/2023,standard class,jp-11135,james peterman,corporate,canada,st. john's,...,a0a,east,fur-fu-10002501,furniture,furnishings,nu-dell executive frame,99.120,8,0.0,35.4144


L'ispezione mostra chiaramente 2 diverse righe ripetute. Il fatto che l'id sia consecutivo è ambiguo. Potrebbe trattarsi di un operatore che per qualche motivo ha inserito una seconda volta un record, convinto che il primo sia andato in errore. Non immagino motivi per i quali un ordine debba essere diviso in 2 righe dello stesso esatto valore. Trattandosi inoltre di due stati completamente separati, non si può nemmeno attribuire ad una abitudine dello stesso cliente o venditore.

Detto questo, e considerato che si tratta di sole 2 righe su oltre 10000 procedo al drop di queste.

In [74]:
df_analizzato = df_sup_normalized.drop_duplicates(subset=colonne_da_controllare, keep='first')
#   df_analizzato.info()

Ora le righe sono 10191, 3 in meno rispetto al CSV orginale (1 persa per la riga con valore non convertibile in numero nella colonna Profit, e 2 perse perchè duplicate).

Terminata l'analisi preliminare, procedo a spacchettare nelle diverse tabelle. Il primo passo per creare la dimensione anagrafiche clienti, e verificare che il customer ID sia collegato ad un unico nome e segment.
Per farlo raggruppo in base al Customer ID conteggiando i valori unici associati di nome cliente e segmento. Se la somma supera 1, significa che customer ID non è univoco.

In [75]:
controllo_customer_id = (df_analizzato.groupby("Customer ID").agg(
    nome_cliente = ("Customer Name", "nunique"),
    segment = ("Segment", "nunique")
))
nome_cliente_non_univoco = (controllo_customer_id["nome_cliente"] > 1).sum()
segment_non_univoco = (controllo_customer_id["segment"] > 1).sum()

print(f"Customer ID con nome cliente non univoco = {nome_cliente_non_univoco}")
print(f"Customer ID con segment non univoco = {segment_non_univoco}")

Customer ID con nome cliente non univoco = 0
Customer ID con segment non univoco = 0


Lo stesso tipo di controllo per l'anagrafica dei prodotti.
Raggruppo per product_Id, e verifico se per ognuno le corrispondenze sono uniche (nunique).
Colonne interessate: Product ID, Category, Sub-Category, Product Name

In [76]:
controllo_product_id = (df_analizzato.groupby("Product ID").agg(
    categoria = ("Category", "nunique"),
    sotto_categoria = ("Sub-Category", "nunique"),
    nome_prodotto = ("Product Name", "nunique")
))
categoria_non_univoco = (controllo_product_id["categoria"] > 1).sum()
sotto_categora_non_univoco = (controllo_product_id["sotto_categoria"] > 1).sum()
nome_prodotto_non_univoco = (controllo_product_id["nome_prodotto"] > 1).sum()

print(f"Product ID con categoria non univoco = {categoria_non_univoco}")
print(f"Product ID con sotto categoria non univoco = {sotto_categora_non_univoco}")
print(f"Product ID con nome prodotto non univoco = {nome_prodotto_non_univoco}")


Product ID con categoria non univoco = 0
Product ID con sotto categoria non univoco = 0
Product ID con nome prodotto non univoco = 32


L'analisi dimostra che lo stesso Product ID può avere diversi nomi prodotto. Con un filtro estrapolo i Product ID con nome prodotto multipli.
Sulla tabella raggruppata per product_id (controllo_product_id), creo una maschera per filtrare tutti i product_id con nome_prodotto maggiore di 1 (ricordiamo che nome_prodotto è la colonna che conteggia il numero di valori unici).

In [77]:
mask_product_id_multipli = controllo_product_id["nome_prodotto"] > 1
product_id_multipli = controllo_product_id[mask_product_id_multipli]
print(f"Product ID con più di un nome prodotto associato:\n {product_id_multipli}")

Product ID con più di un nome prodotto associato:
                  categoria  sotto_categoria  nome_prodotto
Product ID                                                
fur-bo-10002213          1                1              2
fur-ch-10001146          1                1              2
fur-fu-10001473          1                1              2
fur-fu-10004017          1                1              2
fur-fu-10004091          1                1              2
fur-fu-10004270          1                1              2
fur-fu-10004848          1                1              2
fur-fu-10004864          1                1              2
off-ap-10000576          1                1              2
off-ar-10001149          1                1              2
off-bi-10002026          1                1              2
off-bi-10004632          1                1              2
off-bi-10004654          1                1              2
off-pa-10000357          1                1              2
off-p

Esamino uno dei Product ID per vedere l'anomalia

In [78]:
anomalia = df_analizzato[df_analizzato["Product ID"] == "FUR-BO-10002213"]
#   print(anomalia)

L' analisi rivela due product Name completamente diversi, dal costo nettamente diverso, associati allo stesso product ID. In questo caso su 10 righe, 4 hanno nome articolo differente.
Decido di esplorare un altro product ID per avere un altro campione

In [79]:
anomalia_2 = df_analizzato[df_analizzato["Product ID"] == "TEC-PH-10001795"]
# print(anomalia_2)

Qui, su 5 righe, solo una riporta un product name differente. Anche qui però il prezzo del prodotto differisce di molto.
E' necessario capire quanto impattano queste righe sulla tabella finale. Creo quindi una maschera per tutti quei product ID con nome prodotto non univoco, usando il dataframe "product_id_multipli" come indice.


In [80]:
index_product_id_multipli = product_id_multipli.index
maschera_id_multipli = df_analizzato["Product ID"].isin(index_product_id_multipli)

# applico la maschera al dataframe completo #

elenco_product_ID_multipli = df_analizzato[maschera_id_multipli]
print("Numero di righe per product ID con 'product name' non univoco:")
print(elenco_product_ID_multipli.shape[0])

Numero di righe per product ID con 'product name' non univoco:
343


Il risultato conta 343 righe. Questo rappresenta il caso peggiore, non è detto che tutte le righe coinvolte siano errate.

Ricordiamo, ad esempio, che nella prima analisi di anomalia, su 10 righe osservate, 6 appartenevano a un Product Name e 4 a un altro.

È quindi possibile che una parte delle 343 righe sia comunque corretta.
Al momento, però, non è possibile stabilire quale dei due Product Name associati allo stesso Product ID sia quello attendibile.

Rimangono quindi da chiarire alcuni punti:

Quanti Product Name differenti possono essere associati allo stesso Product ID? (ad esempio: il massimo osservato è sempre 2?)
Come identificare il Product Name corretto quando uno stesso Product ID è associato a più valori?

Per verificare il primo punto, riutilizzo la maschera 'mask_product_id_multipli' ma con filtro > 2.

In [81]:
mask_product_id_multipli_2 = controllo_product_id["nome_prodotto"] > 2
product_id_multipli_2 = controllo_product_id[mask_product_id_multipli_2]
print(f"Product ID con più di un nome prodotto associato:\n {product_id_multipli_2.shape[0]}")

Product ID con più di un nome prodotto associato:
 0


Il risultato è 0, significa quindi che il massimo di product name associato ad ogni product ID è 2.
Ora è necessario decidere quale regola applicare per scegliere il product name da mantenere.
Per farlo imposto 2 scenari:
1) mantengo il product name con maggiore occorrenza
2) mantengo il product name con minore occorrenza
e vedere l'impatto sul database originale.

Questo non solo come numero di righe, ma anche come valore di fatturato, dal momento che il problema evidenziato dal cliente (descritto all' inizio) è più legato alla marginalità, che al numero di prodotti venduti.

Imposto quindi una nuova tabella aggragata per 'product ID' E 'product name' partendo dal dataframe filtrato sui Product ID non univoci.

In [82]:
df_occorrenze = elenco_product_ID_multipli.groupby(["Product ID", "Product Name"]).agg(
    occorrenze_product_name = ("Product Name", "count"),
    total_sales = ("Sales", "sum"),
    total_profit = ("Profit", "sum")
)

df_occorrenze = df_occorrenze.reset_index()

#   pd.set_option("display.max_rows", None)
#   display(df_occorrenze)
#   pd.reset_option("display.max_rows")


L'analisi delle occorrenze, del fatturato e del profitto associati ai Product ID anomali non evidenzia un criterio univoco per determinare quale Product Name sia corretto. In alcuni casi il nome più frequente genera il maggior fatturato, in altri accade il contrario. Stessa cosa con il profitto.

Anche a livello di pecentuali di occorrenze, non si evidenziano schemi particolari (passiamo da rapporti di 50 / 50 a rapporti 10 / 90), rendendo quindi difficile operare una scelta. Per quanto i 343 record rappresentino circa il 3% dei dati, in assenza della possibilità di interagire con il cliente, decido di annotare l'anomalia e proseguire oltre. 

A scopo puramente dimostrativo, proseguo presentando i risultati dello scenario A (mantengo i dati per le occorrenze maggiori) e scenario B (mantengo le occorrenze minori)

Per farlo riordino la tabella aggregata "df_occorrenze" per "product ID" (per mantenerli vicini visivamente), poi per "product name".
La scelta dei criteri secondari (Sales e Profit) è arbitraria e utilizzata esclusivamente per gestire eventuali casi di parità nelle occorrenze.

In [83]:
# Caso A : Mantengo i product name con più occorrenze, in caso di parità mantengo quello con sales e profit maggiori #

df_occorrenze_A = df_occorrenze.sort_values(by=["Product ID", "occorrenze_product_name", "total_sales", "total_profit"], 
                                            ascending=[True, False, False, False])

#   pd.set_option("display.max_rows", None)
#   display(df_occorrenze)
#   pd.reset_option("display.max_rows")

df_occorrenze_A = df_occorrenze_A.drop_duplicates(subset=["Product ID"], keep="first")

print(f"Numero righe df_occorrenze_A: {df_occorrenze_A.shape[0]}")
df_occorrenze_A.head()


Numero righe df_occorrenze_A: 32


,Product ID,Product Name,occorrenze_product_name,total_sales,total_profit
0,fur-bo-10002213,dmi eclipse executive suite bookcases,6,11046.609,90.1764
3,fur-ch-10001146,"global value mid-back manager's chair, gray",10,1978.925,197.8925
4,fur-fu-10001473,dax wood document frame,9,477.804,126.3160
7,fur-fu-10004017,tenex contemporary contur chairmats for low an...,4,1010.782,150.5420
9,fur-fu-10004091,"howard miller 13"" diameter goldtone round wall...",6,1042.068,294.3138


In [84]:
# Caso B : Mantengo i product name con meno occorrenze, in caso di parità mantengo quello con sales e profit maggiori #

df_occorrenze_B = df_occorrenze.sort_values(by=["Product ID", "occorrenze_product_name", "total_sales", "total_profit"], 
                                            ascending=[True, True, True, True])

#   pd.set_option("display.max_rows", None)
#   display(df_occorrenze)
#   pd.reset_option("display.max_rows")

df_occorrenze_B = df_occorrenze_B.drop_duplicates(subset=["Product ID"], keep="first")

print(f"Numero righe df_occorrenze_B: {df_occorrenze_B.shape[0]}")
df_occorrenze_B.head()

Numero righe df_occorrenze_B: 32


,Product ID,Product Name,occorrenze_product_name,total_sales,total_profit
1,fur-bo-10002213,"sauder forest hills library, woodland oak finish",4,1875.034,-70.4900
2,fur-ch-10001146,"global task chair, black",5,463.099,-80.4062
5,fur-fu-10001473,"eldon executive woodline ii desk accessories, ...",5,361.872,50.0087
6,fur-fu-10004017,"executive impressions 13"" chairman wall clock",3,253.800,72.3330
8,fur-fu-10004091,"eldon 200 class desk accessories, black",2,75.360,28.6368


Per effettuare i report dei due casi, procedo per prima cosa a filtrare tutti i casi anomali dal dataframe originale ("df_analizzato" che è la versione normalizzata), che mi servirà poi anche per procedere con l'analisi dati.
A questo farò un merge del caso A e poi del caso B.

In [85]:
# Creazione dataframe senza casi anomali #

maschera_product_id_multipli = df_analizzato["Product ID"].isin(index_product_id_multipli)
df_finale = df_analizzato[~maschera_product_id_multipli].copy()
print(f"Numero righe df_finale: {df_finale.shape[0]}")

sales_profit = (df_finale.groupby("Product ID").agg(
    total_sales = ("Sales", "sum"),
    total_profit = ("Profit", "sum")
    ))
print(f"Sales totali senza anomalie: {(sales_profit["total_sales"].sum()).round(2)}\nProfit totale senza anomalie: {(sales_profit["total_profit"].sum()).round(2)}")

Numero righe df_finale: 9848
Sales totali senza anomalie: 2227885.25
Profit totale senza anomalie: 278940.7


In [86]:
# Dataframe caso A #
# Creazione tupla con Product ID e Product Name validi, partendo da df_occorrenze_B (quelli da filtrare) #
coppie_anomale_B = set(zip(df_occorrenze_B["Product ID"], df_occorrenze_B["Product Name"]))
maschera_occorrenze_B = (df_analizzato[["Product ID", "Product Name"]].apply(tuple, axis=1).isin(coppie_anomale_B))

df_finale_caso_A = df_analizzato[~maschera_occorrenze_B]
print(f"Numero righe df_finale_caso_A: {df_finale_caso_A.shape[0]}")
print(f"Righe aggiunte caso A: {df_finale_caso_A.shape[0] - df_finale.shape[0]}")

sales_profit_caso_A = (df_finale_caso_A.groupby("Product ID").agg(
    total_sales = ("Sales", "sum"),
    total_profit = ("Profit", "sum")
    ))
print(f"Sales totali caso A: {(sales_profit_caso_A["total_sales"].sum()).round(2)}\nProfit totale caso A: {(sales_profit_caso_A["total_profit"].sum()).round(2)}")

Numero righe df_finale_caso_A: 10062
Righe aggiunte caso A: 214
Sales totali caso A: 2295721.19
Profit totale caso A: 286162.88


In [87]:
# Dataframe caso B #
# Creazione tupla con Product ID e Product Name validi, partendo da df_occorrenze_A (quelli da filtrare) #
coppie_anomale_A = set(zip(df_occorrenze_A["Product ID"], df_occorrenze_A["Product Name"]))
maschera_occorrenze_A = (df_analizzato[["Product ID", "Product Name"]].apply(tuple, axis=1).isin(coppie_anomale_A))

df_finale_caso_B = df_analizzato[~maschera_occorrenze_A]
print(f"Numero righe df_finale_caso_B: {df_finale_caso_B.shape[0]}")
print(f"Righe aggiunte caso B: {df_finale_caso_B.shape[0] - df_finale.shape[0]}")

sales_profit_caso_B = (df_finale_caso_B.groupby("Product ID").agg(
    total_sales = ("Sales", "sum"),
    total_profit = ("Profit", "sum")
    ))
print(f"Sales totali caso B: {(sales_profit_caso_B['total_sales'].sum()).round(2)}\nProfit totale caso B: {(sales_profit_caso_B['total_profit'].sum()).round(2)}")

Numero righe df_finale_caso_B: 9977
Righe aggiunte caso B: 129
Sales totali caso B: 2257983.93
Profit totale caso B: 285047.35


Riassunto 3 casistiche:
### No anomalie
Numero righe df_finale: **9848**

Sales totali senza anomalie: **2227885.25**

Profit totale senza anomalie: **278940.7**

### Caso A
Numero righe df_finale_caso_A: **10062**

Righe aggiunte caso A: 214

Sales totali caso A: **2295721.19**

Profit totale caso A: **286162.88**


### Caso B
Numero righe df_finale_caso_B: **9977**

Righe aggiunte caso B: 129

Sales totali caso B: **2257983.93**

Profit totale caso B: **285047.35**


Procedo ora con la suddivisione in tabelle DIM e FACT basandomi sul dataframe filtrato dai casi anomali

In [88]:
# Creazione del percorso di output
folder_output = os.path.join("..", "data", "processed")
os.makedirs(folder_output, exist_ok=True)

In [89]:
# Funzione per esportare le diverse tabelleDIM_Date
def dataframe_export(dataframe, folder, nome):
    percorso_dataframe = os.path.join(folder, nome)
    dataframe_reset_index = dataframe.reset_index(drop=True)
    dataframe_reset_index.to_csv(percorso_dataframe, index=False)

    print(f"File esportato: {percorso_dataframe}")

In [90]:
# Creazione DIM_Customers #

DIM_Customers = df_finale[["Customer ID", "Customer Name", "Segment", "Country/Region", "City", "State/Province", "Postal Code", "Region"]].drop_duplicates(subset=["Customer ID"], keep="first")

# Creazione DIM_Products #

DIM_Products = df_finale[["Product ID", "Product Name", "Category", "Sub-Category"]].drop_duplicates(subset=["Product ID"], keep="first")


Creazione tabella dim_date (Calendario)

In [91]:
# Conversione per creazione tabella DIM_Date
df_finale["Order Date"] = pd.to_datetime(df_finale["Order Date"])
df_finale["Ship Date"] = pd.to_datetime(df_finale["Ship Date"])


# Prendo tutte le date presenti in Order Date e Ship Date, le mette in un’unica colonna chiamata date, 
# elimina i duplicati e crea la tabella DIM_Date.
DIM_Date = pd.DataFrame({
    "date": pd.concat([df_finale["Order Date"], df_finale["Ship Date"]]).drop_duplicates()
})


# Ordino le date in ordine cronologico e ricreo l’indice da zero.
DIM_Date = DIM_Date.sort_values("date").reset_index(drop=True)

# Crea una chiave numerica per ogni data (date_id) da usare nello schema a stella.
DIM_Date["date_id"] = DIM_Date["date"].dt.strftime("%Y%m%d").astype(int)


DIM_Date["year"] = DIM_Date["date"].dt.year
DIM_Date["month"] = DIM_Date["date"].dt.month
DIM_Date["month_name"] = DIM_Date["date"].dt.month_name()
DIM_Date["quarter"] = DIM_Date["date"].dt.quarter
DIM_Date["day"] = DIM_Date["date"].dt.day
DIM_Date["day_of_week"] = DIM_Date["date"].dt.day_name()

# Riordino le colonne.
DIM_Date = DIM_Date[
    ["date_id", "date", "year", "quarter", "month", "month_name", "day", "day_of_week"]
]

Creazione tabella FACT_Sales

Dalla df_finale prenderemo:

Chiavi

Order ID
Customer ID
Product ID
order_date_id
ship_date_id

Misure

Sales
Quantity
Discount
Profit

In [92]:
# Creo le due chiavi Order_Date_ID e Ship_Date_ID nella tabella dei fatti (fact_table).
# Uso lo stesso principio utilizzato nell DIM_Date, ma qui sto creando un ID per ordine e un ID per spedizione legato ad ogni riga.
# Servirà nello schema a stella per collegare la tabella dei fatti con la tabella delle date.

df_finale["order_date_id"] = (
    df_finale["Order Date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

df_finale["ship_date_id"] = (
    df_finale["Ship Date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# Creo la tabella dei fatti (fact_table) con le colonne richieste e le chiavi surrogate.
FACT_Sales = df_finale[
    [
        "Order ID",
        "Customer ID",
        "Product ID",
        "order_date_id",
        "ship_date_id",
        "Sales",
        "Quantity",
        "Discount",
        "Profit",
    ]
].copy()

In [93]:
# Pulizia nomi colonne per postgreSQL

DIM_Customers.columns = [
    "customer_id",
    "customer_name",
    "segment",
    "country_region",
    "city",
    "state_province",
    "postal_code",
    "region"
]

DIM_Products.columns = [
    "product_id",
    "product_name",
    "category",
    "sub_category",
]

DIM_Date.columns = [
    "date_id",
    "date",
    "year",
    "quarter",
    "month",
    "month_name",
    "day",
    "day_of_week"
]

FACT_Sales.columns = [
    "order_id",
    "customer_id",
    "product_id",
    "order_date_id",
    "ship_date_id",
    "sales",
    "quantity",
    "discount",
    "profit"
]

# Esportazione tabelle

dataframe_export(DIM_Customers, folder_output, "DIM_Customers.csv")
dataframe_export(DIM_Products, folder_output, "DIM_Products.csv")
dataframe_export(DIM_Date, folder_output, "DIM_Date.csv")
dataframe_export(FACT_Sales, folder_output, "FACT_Sales.csv")


File esportato: ..\data\processed\DIM_Customers.csv
File esportato: ..\data\processed\DIM_Products.csv
File esportato: ..\data\processed\DIM_Date.csv
File esportato: ..\data\processed\FACT_Sales.csv


Verifica che ogni chiave presente nella Fact esista nella rispettiva Dimensione

In [94]:
# Customer key check
set(FACT_Sales["customer_id"]) - set(DIM_Customers["customer_id"])

# Product key check
#set(FACT_Sales["product_id"]) - set(DIM_Products["product_id"])

# Order date key check
#set(FACT_Sales["order_date_id"]) - set(DIM_Date["date_id"])

# Ship date key check
#set(FACT_Sales["ship_date_id"]) - set(DIM_Date["date_id"])



set()

Eseguendo i controlli il terminale resitutisce sempre un set vuoto, quindi tutte le dimensioni hanno una corispondeza nella tabella FACT

In [95]:
DIM_Customers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 804 entries, 0 to 9555
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customer_id     804 non-null    object
 1   customer_name   804 non-null    object
 2   segment         804 non-null    object
 3   country_region  804 non-null    object
 4   city            804 non-null    object
 5   state_province  804 non-null    object
 6   postal_code     804 non-null    object
 7   region          804 non-null    object
dtypes: object(8)
memory usage: 56.5+ KB
